In [189]:
import pandas as pd
import geopandas as gpd
import numpy as np

from tqdm.notebook import tqdm
import requests, cma

import pickle, os

In [190]:
# reference_path = "../../../results/road/freeflow/reference_api.parquet"
# output_path = "../../../results/road/freeflow/calibration_cache_api.pickle"

reference_path = "../../../results/road/freeflow/reference_survey.parquet"
output_path = "../../../results/road/freeflow/calibration_cache_survey.pickle"

routing_endpoint = "http://sma.univ-eiffel.fr:18054/router/road"
# routing_endpoint = "http://localhost:8054/router/road"
departure_time = 4 * 3600
maximum_batch_size = 400

In [191]:
# Load reference
df_reference = pd.read_parquet(reference_path)

In [192]:
# Prepare requests
df_reference["request_index"] = np.arange(len(df_reference))

# Convert to requests
request_list = []

for index, row in df_reference.iterrows():
    request_list.append({
        "request_index": int(row["request_index"]),
        "origin_x": row["origin_x"],
        "origin_y": row["origin_y"],
        "destination_x": row["destination_x"],
        "destination_y": row["destination_y"],
        "departure_time_s": departure_time,
        "consider_parallel_links": True
    })

In [193]:
# Prepare querying
def query_requests(request_list, settings):
    df_response = []
    batch_index = 0

    while batch_index * maximum_batch_size < len(request_list):
        batch = request_list[batch_index * maximum_batch_size : (batch_index + 1) * maximum_batch_size]

        response = requests.post(routing_endpoint, verify=False, json = {
            "batch": batch,
            "freespeed": settings
        })

        df_response.append(pd.DataFrame.from_records(response.json()))
        batch_index += 1

    return pd.concat(df_response)

In [194]:
# Define calibration variables
variables = [
    { "name": "major_factor", "initial": 1.0, "bounds": (1.0, np.inf) },
    { "name": "intermediate_factor", "initial": 1.0, "bounds": (1.0, np.inf) },
    { "name": "minor_factor", "initial": 1.0, "bounds": (1.0, np.inf) },
    { "name": "major_crossing_penalty_s", "initial": 4, "bounds": (0.0, np.inf) },
    { "name": "minor_crossing_penalty_s", "initial": 0.0, "bounds": (0.0, np.inf) },
]

In [195]:
# Extend with index information for CMA-ES evaluation
variables_map = { v["name"]: v for v in variables }

active_index = 0

# First find active variables
for variable in variables:
    variables_map[variable["name"]] = variable
    variable["index"] = active_index
    active_index += 1

In [196]:
# Define the optimization objective
def calculate_objective(df_evaluation, df_reference):
    df_reference = df_reference[["request_index", "reference_travel_time_s", "weight"]].copy()

    df_evaluation = df_evaluation[["request_index", "total_travel_time_min"]].copy()
    df_evaluation["evaluation_travel_time_s"] = df_evaluation["total_travel_time_min"] * 60.0
    df_evaluation = df_evaluation[["request_index", "evaluation_travel_time_s"]]

    df_comparison = pd.merge(df_reference, df_evaluation, on = "request_index")
    df_comparison["difference_s"] = np.abs(df_comparison["evaluation_travel_time_s"] - df_comparison["reference_travel_time_s"])

    if False:
        mean = np.sum(df_comparison["reference_travel_time_s"] * df_comparison["weight"]) / df_comparison["weight"].sum()
        ss_residuals = np.sum(df_comparison["weight"] * df_comparison["difference_s"]**2)
        ss_total = np.sum(df_comparison["weight"] * (df_comparison["reference_travel_time_s"] - mean)**2)
        R2 = 1 - ss_residuals / ss_total
        
        return (
            1 - R2, df_comparison
        )
    
    return (
        np.sum(df_comparison["weight"] * np.abs(df_comparison["difference_s"])) / df_comparison["weight"].sum(), 
        df_comparison
    )

In [197]:
# Prepare function to convert CMA-ES' candidate to freespeed settings
def prepare_settings(values):
    settings = {}
    
    for variable in variables:
        settings[variable["name"]] = values[variable["index"]]
    
    return settings

In [198]:
# Prepare bounds and initial values
initial = []
bounds = [[], []]

for variable in variables:
    initial.append(variable["initial"])
    bounds[0].append(variable["bounds"][0])
    bounds[1].append(variable["bounds"][1])

In [199]:
query_requests(request_list[:5], prepare_settings(initial))


,request_index,in_vehicle_distance_km,in_vehicle_time_min,access_time_min,egress_time_min,access_distance_km,egress_distance_km,arrivalTime_s,total_travel_time_min
0,0,31.412478,20.339664,0.354031,1.957208,0.019608,0.108399,15641.621722,22.650904
1,1,31.547420,22.867549,0.685685,0.870974,0.037976,0.048239,15813.194050,24.424208
2,2,8.449291,8.778688,1.692092,1.030397,0.093716,0.057068,15028.246774,11.501176
3,3,2.571826,3.131570,1.230178,0.385106,0.068133,0.021329,14661.704898,4.746854
4,4,3.007590,3.724288,1.315855,0.385106,0.072878,0.021329,14702.408588,5.425249


In [200]:
# Test connection
assert len(query_requests(request_list[:5], prepare_settings(initial))) == 5

In [ ]:
# Configure CMA-ES
seed = 1000
sigma = 1.0
iterations = 200

options = cma.CMAOptions()
options.set("bounds", bounds)
options.set("seed", seed)

algorithm = cma.CMAEvolutionStrategy(initial, sigma, options)

# Load cached data for previous iterations
history = []

if os.path.exists(output_path):
    with open(output_path, "rb") as f:
        history = pickle.load(f)

        algorithm.feed_for_resume(
            [h["candidate"] for h in history[1:]], # first one is initial
            [h["objective"] for h in history[1:]]
        )

# Perform a new batch of iterations
for iteration in range(iterations):
    initial_evaluation = len(history) == 0
    candidates = [initial]

    if not initial_evaluation:
        candidates = algorithm.ask()

    objectives = []

    for candidate in candidates:
        settings = prepare_settings(candidate)
        df_response = query_requests(request_list, settings)
        objective, df_comparison = calculate_objective(df_response, df_reference)

        objectives.append(objective)

        history.append({
            "candidate": candidate,
            "settings": settings,
            "objective": objective,
            "evaluation": df_comparison,
            "initial": initial_evaluation
        })

    if not initial_evaluation:
        algorithm.tell(candidates, objectives)
        algorithm.disp()

    # Save after a successful CMA-ES iteration
    with open(output_path, "wb+") as f:
        pickle.dump(history, f)

(4_w,8)-aCMA-ES (mu_w=2.6,w_1=52%) in dimension 5 (seed=1000, Fri Jan 24 15:47:26 2025)


Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1      8 4.774098402077937e+02 1.0e+00 8.36e-01  8e-01  8e-01 0:08.4
    2     16 5.314650036817377e+02 1.3e+00 7.71e-01  6e-01  8e-01 0:17.2
    3     24 5.308027029566758e+02 1.6e+00 6.86e-01  6e-01  7e-01 0:27.7
    4     32 5.333865837194286e+02 1.6e+00 6.34e-01  5e-01  6e-01 0:37.1
    5     40 5.351474489061253e+02 1.7e+00 5.63e-01  4e-01  6e-01 0:47.8
    6     48 5.510075707518207e+02 2.0e+00 4.87e-01  3e-01  5e-01 0:56.6
    7     56 4.830624946423966e+02 2.0e+00 5.42e-01  3e-01  6e-01 1:05.5
    8     64 4.915949965741103e+02 2.4e+00 5.99e-01  3e-01  7e-01 1:13.7
    9     72 4.705911024128221e+02 2.5e+00 6.33e-01  3e-01  8e-01 1:22.6
   11     88 4.895412585677265e+02 3.5e+00 7.01e-01  3e-01  1e+00 1:38.5
   13    104 4.778105938993796e+02 3.8e+00 7.66e-01  3e-01  1e+00 1:53.9
   15    120 4.851048949967535e+02 4.3e+00 6.24e-01  2e-01  9e-01 2:10.2
   17    136 4.795641497826986e+02 4.8e+00 5.34e-01 